# PRINCE Clifford / Toffoli Resource Notebook

This notebook mirrors the full PRINCE circuit built in `Main.ipynb`, but focuses only on resource estimation. The raw circuit is restricted to Clifford gates plus Toffoli: `x`, `cx`, and `ccx`. Barriers and measurements are ignored for resource counts.

It reports two views:

1. Clifford + Toffoli view: counts raw `x`, `cx`, and `ccx` gates.
2. Clifford + T view: decomposes each Toffoli into the standard Qiskit basis `h`, `cx`, `t`, `tdg`, then counts Clifford gates, T gates, T-depth, and total depth.


In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

from Helpers import xor_constant_into_register, xor_register_into_register
from Round import quantum_round, quantum_inverse_round, classical_round, classical_inverse_round
from Sbox import quantum_s_layer, quantum_s_layer_inv, classical_s_layer, classical_s_layer_inv
from Mbox import quantum_m_layer, quantum_m_layer_inv, classical_m_layer, classical_m_layer_inv


## Constants and Key Schedule

These are copied from `Main.ipynb` so this resource notebook can run independently.

In [2]:
ROUND_CONSTANTS = [
    0x0000000000000000,
    0x13198A2E03707344,
    0xA4093822299F31D0,
    0x082EFA98EC4E6C89,
    0x452821E638D01377,
    0xBE5466CF34E90C6C,
    0x7EF84F78FD955CB1,
    0x85840851F1AC43AA,
    0xC882D32F25323C54,
    0x64A51195E0E3610D,
    0xD3B5A399CA0C2399,
    0xC0AC29B7C97C50DD,
]

RC0 = ROUND_CONSTANTS[0]
MASK64 = (1 << 64) - 1


def prince_key_schedule(K_128bit):
    k0 = (K_128bit >> 64) & MASK64
    k1 = K_128bit & MASK64
    ror1 = ((k0 >> 1) | (k0 << 63)) & MASK64
    k0_prime = ror1 ^ (k0 >> 63)
    return k0, k1, k0_prime


## Circuit Builder

`Main.ipynb` builds the full PRINCE encryption circuit with initialization, barriers, and measurement. For resource estimation, the default here builds the algorithmic circuit only: no input-loading X gates and no final measurement. This avoids making the resource estimate depend on a particular plaintext/key Hamming weight.

In [3]:
def build_prince_clifford_toffoli_circuit(
    plaintext_64bit=0,
    K_128bit=0,
    include_initialization=False,
    include_measurement=False,
):
    k0, k1, k0_prime = prince_key_schedule(K_128bit)

    state = QuantumRegister(64, name="state")
    anc = QuantumRegister(16 * 14, name="anc")
    qk0 = QuantumRegister(64, name="k0")
    qk1 = QuantumRegister(64, name="k1")
    qk0p = QuantumRegister(64, name="k0p")

    registers = [state, anc, qk0, qk1, qk0p]
    creg = None
    if include_measurement:
        creg = ClassicalRegister(64, name="out")
        registers.append(creg)

    qc = QuantumCircuit(*registers)

    if include_initialization:
        for bit in range(64):
            if (plaintext_64bit >> bit) & 1:
                qc.x(state[bit])
            if (k0 >> bit) & 1:
                qc.x(qk0[bit])
            if (k1 >> bit) & 1:
                qc.x(qk1[bit])
            if (k0_prime >> bit) & 1:
                qc.x(qk0p[bit])

    xor_register_into_register(qc, qk0, state)
    xor_register_into_register(qc, qk1, state)
    xor_constant_into_register(qc, state, RC0)

    for i in range(1, 6):
        quantum_round(i, qc, state, anc, qk1, ROUND_CONSTANTS[i])

    quantum_s_layer(qc, state, anc)
    quantum_m_layer(qc, state)
    quantum_s_layer_inv(qc, state, anc)

    for i in range(6, 11):
        quantum_inverse_round(i, qc, state, anc, qk1, ROUND_CONSTANTS[i])

    xor_constant_into_register(qc, state, ROUND_CONSTANTS[11])
    xor_register_into_register(qc, qk1, state)
    xor_register_into_register(qc, qk0p, state)

    if include_measurement:
        qc.measure(state, creg)

    return qc


## Resource Helpers

`x` and `cx` are Clifford. `ccx` is Toffoli. For Clifford+T reporting, this notebook uses Qiskit's exact Toffoli decomposition: one `ccx` becomes `2 h + 6 cx + 4 t + 3 tdg`.

In [4]:
IGNORED_OPS = {"barrier", "measure"}
RAW_CLIFFORD_OPS = {"x", "cx"}
RAW_ALLOWED_OPS = RAW_CLIFFORD_OPS | {"ccx"}
CLIFFORD_T_BASIS = ["h", "x", "z", "s", "sdg", "cx", "t", "tdg"]
CLIFFORD_T_CLIFFORD_OPS = {"h", "x", "z", "s", "sdg", "cx"}
T_OPS = {"t", "tdg"}


def op_counts_without_ignored(qc):
    return {name: count for name, count in dict(qc.count_ops()).items() if name not in IGNORED_OPS}


def assert_clifford_toffoli_only(qc):
    ops = set(op_counts_without_ignored(qc))
    unexpected = sorted(ops - RAW_ALLOWED_OPS)
    if unexpected:
        raise ValueError(f"Unexpected non Clifford+Toffoli gates: {unexpected}")
    return True


def depth_without_ignored(qc):
    return qc.depth(filter_function=lambda inst: inst.operation.name not in IGNORED_OPS)


def raw_clifford_toffoli_resources(qc):
    assert_clifford_toffoli_only(qc)
    ops = op_counts_without_ignored(qc)
    x_count = ops.get("x", 0)
    cx_count = ops.get("cx", 0)
    ccx_count = ops.get("ccx", 0)
    return {
        "total_qubits": qc.num_qubits,
        "x_gates": x_count,
        "cx_gates": cx_count,
        "toffoli_gates": ccx_count,
        "total_clifford_gates_raw": x_count + cx_count,
        "total_gates_raw": x_count + cx_count + ccx_count,
        "logical_depth_raw": depth_without_ignored(qc),
    }


def to_clifford_t(qc, optimization_level=0):
    return transpile(qc, basis_gates=CLIFFORD_T_BASIS, optimization_level=optimization_level)


def clifford_t_resources(qc, optimization_level=0):
    converted = to_clifford_t(qc, optimization_level=optimization_level)
    ops = op_counts_without_ignored(converted)
    clifford_count = sum(ops.get(name, 0) for name in CLIFFORD_T_CLIFFORD_OPS)
    t_count = sum(ops.get(name, 0) for name in T_OPS)
    t_depth = converted.depth(filter_function=lambda inst: inst.operation.name in T_OPS)
    return {
        "total_qubits": converted.num_qubits,
        "total_clifford_gates": clifford_count,
        "total_t_gates": t_count,
        "t_depth": t_depth,
        "total_depth": depth_without_ignored(converted),
        "gate_wise_counts": ops,
    }


## Functional Tests

These tests mirror the checks in `Main.ipynb`: build the quantum circuit, simulate it, and compare against the classical PRINCE implementation. They run both the raw Clifford+Toffoli circuit and the Clifford+T decomposed circuit.

In [5]:
def classical_prince_encrypt(plaintext_64bit, K_128bit):
    k0, k1, k0_prime = prince_key_schedule(K_128bit)

    state = plaintext_64bit & MASK64
    state ^= k0
    state ^= k1
    state ^= RC0

    for i in range(1, 6):
        state = classical_round(i, state, k1, ROUND_CONSTANTS[i])

    state = classical_s_layer(state)
    state = classical_m_layer(state)
    state = classical_s_layer_inv(state)

    for i in range(6, 11):
        state = classical_inverse_round(i, state, k1, ROUND_CONSTANTS[i])

    state ^= ROUND_CONSTANTS[11]
    state ^= k1
    state ^= k0_prime
    return state & MASK64


def simulate_output(qc):
    sim = AerSimulator(method="matrix_product_state")
    result = sim.run(qc, shots=1).result()
    counts = result.get_counts()
    return int(next(iter(counts)), 2)


def run_functional_test(plaintext_64bit, K_128bit, test_name):
    expected = classical_prince_encrypt(plaintext_64bit, K_128bit)
    raw_qc = build_prince_clifford_toffoli_circuit(
        plaintext_64bit,
        K_128bit,
        include_initialization=True,
        include_measurement=True,
    )
    raw_result = simulate_output(raw_qc)

    converted_qc = to_clifford_t(raw_qc, optimization_level=0)
    converted_result = simulate_output(converted_qc)

    raw_ok = raw_result == expected
    converted_ok = converted_result == expected
    print(f"{test_name}")
    print(f"  plaintext        : 0x{plaintext_64bit:016X}")
    print(f"  key              : 0x{K_128bit:032X}")
    print(f"  expected         : 0x{expected:016X}")
    print(f"  raw circuit      : 0x{raw_result:016X}  {'PASS' if raw_ok else 'FAIL'}")
    print(f"  Clifford+T       : 0x{converted_result:016X}  {'PASS' if converted_ok else 'FAIL'}")
    assert raw_ok and converted_ok


test_vectors = [
    (0x0000000000000000, 0x00000000000000000000000000000000, "Test 1: all-zero plaintext and key"),
    (0x1111111111111111, 0x0ABCDEFFEDC123456789BA9876543210, "Test 2: non-zero plaintext and key"),
]

for pt, key, name in test_vectors:
    run_functional_test(pt, key, name)


Test 1: all-zero plaintext and key
  plaintext        : 0x0000000000000000
  key              : 0x00000000000000000000000000000000
  expected         : 0xB607FA70A2065323
  raw circuit      : 0xB607FA70A2065323  PASS
  Clifford+T       : 0xB607FA70A2065323  PASS
Test 2: non-zero plaintext and key
  plaintext        : 0x1111111111111111
  key              : 0x0ABCDEFFEDC123456789BA9876543210
  expected         : 0x44FBFF21F4BA60E1
  raw circuit      : 0x44FBFF21F4BA60E1  PASS
  Clifford+T       : 0x44FBFF21F4BA60E1  PASS


## Build and Estimate PRINCE Cipher

In [6]:
prince_qc = build_prince_clifford_toffoli_circuit()
raw_resources = raw_clifford_toffoli_resources(prince_qc)
raw_resources


{'total_qubits': 480,
 'x_gates': 1282,
 'cx_gates': 13920,
 'toffoli_gates': 7680,
 'total_clifford_gates_raw': 15202,
 'total_gates_raw': 22882,
 'logical_depth_raw': 10712}

In [7]:
clifford_t_estimate = clifford_t_resources(prince_qc, optimization_level=0)
clifford_t_estimate


{'total_qubits': 480,
 'total_clifford_gates': 72802,
 'total_t_gates': 53760,
 't_depth': 24960,
 'total_depth': 66104,
 'gate_wise_counts': {'cx': 60000,
  't': 30720,
  'tdg': 23040,
  'h': 11520,
  'x': 1282}}

## Compact Tables

These tables are the reviewer-facing form for the cipher-only case. Grover iteration tables need an oracle and diffuser circuit before they can be produced.

In [8]:
try:
    import pandas as pd

    prince_table = pd.DataFrame([raw_resources])
    pprince_style_table = pd.DataFrame([
        {k: v for k, v in clifford_t_estimate.items() if k != "gate_wise_counts"}
    ])

    display(prince_table)
    display(pprince_style_table)
except ImportError:
    print("PRINCE raw Clifford+Toffoli resources:")
    print(raw_resources)
    print("\nPRINCE Clifford+T resources:")
    print({k: v for k, v in clifford_t_estimate.items() if k != "gate_wise_counts"})


,total_qubits,x_gates,cx_gates,toffoli_gates,total_clifford_gates_raw,total_gates_raw,logical_depth_raw
0,480,1282,13920,7680,15202,22882,10712


,total_qubits,total_clifford_gates,total_t_gates,t_depth,total_depth
0,480,72802,53760,24960,66104
